In [1]:
!pip install -q google-generativeai pypdf gradio sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.3/336.3 kB 5.3 MB/s eta 0:00:00


In [2]:

import google.generativeai as genai
from google.colab import userdata
import pypdf
import numpy as np
from sentence_transformers import SentenceTransformer
import gradio as gr
import textwrap
from google.colab import userdata
# Configurar la API Key de Gemini desde Secrets
GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
genai.configure(api_key=GEMINI_API_KEY)



/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [3]:


def extraer_texto_pdf(ruta_pdf):
    lector = pypdf.PdfReader(ruta_pdf)
    texto_completo = ""
    for pagina in lector.pages:
        texto_completo += pagina.extract_text() + "\n"
    return texto_completo

def dividir_en_chunks(texto, tamano_chunk=500, solapamiento=50):
    palabras = texto.split()
    chunks = []
    inicio = 0
    while inicio < len(palabras):
        fin = inicio + tamano_chunk
        chunk = " ".join(palabras[inicio:fin])
        chunks.append(chunk)
        inicio += tamano_chunk - solapamiento
    return chunks

def construir_indice(chunks, modelo_embedding):
    print(f"Generando embeddings para {len(chunks)} chunks...")
    embeddings = modelo_embedding.encode(chunks, show_progress_bar=True)
    return embeddings

def buscar_chunks_relevantes(pregunta, chunks, embeddings, modelo_embedding, top_k=5):
    embedding_pregunta = modelo_embedding.encode([pregunta])
    similitudes = np.dot(embeddings, embedding_pregunta.T).flatten()
    normas = np.linalg.norm(embeddings, axis=1) * np.linalg.norm(embedding_pregunta)
    similitudes_coseno = similitudes / (normas + 1e-10)
    indices_top = np.argsort(similitudes_coseno)[::-1][:top_k]
    chunks_relevantes = [chunks[i] for i in indices_top]
    return chunks_relevantes

print("Funciones definidas OK")

Funciones definidas OK


In [5]:


RUTA_PDF = "/content/ciencia-de-datos-desde-cero-segunda-edicion.pdf"

#  modelo de embeddings (multilingüe para español)
print("Cargando modelo de embeddings...")
modelo_embedding = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
print(" Modelo de embeddings cargado")

# Procesar el PDF
print(f"\nProcesando PDF: {RUTA_PDF}")
texto_pdf = extraer_texto_pdf(RUTA_PDF)
print(f" Texto extraído: {len(texto_pdf)} caracteres")


chunks = dividir_en_chunks(texto_pdf, tamano_chunk=500, solapamiento=50)
print(f" Texto dividido en {len(chunks)} chunks")

embeddings = construir_indice(chunks, modelo_embedding)
print(f" Índice construido con shape: {embeddings.shape}")

Cargando modelo de embeddings...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


 Modelo de embeddings cargado

Procesando PDF: /content/ciencia-de-datos-desde-cero-segunda-edicion.pdf
 Texto extraído: 73005 caracteres
 Texto dividido en 16 chunks
Generando embeddings para 16 chunks...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

 Índice construido con shape: (16, 384)


In [6]:

def responder_pregunta_rag(pregunta, historial=[]):
    """
    Sistema RAG: busca chunks relevantes y genera respuesta con Gemini.
    """
    # 1. Recuperar chunks relevantes
    chunks_relevantes = buscar_chunks_relevantes(
        pregunta, chunks, embeddings, modelo_embedding, top_k=5
    )


    contexto = "\n\n---\n\n".join(chunks_relevantes)


    prompt = f"""Eres un asistente experto que responde preguntas basándose ÚNICAMENTE en el siguiente contexto extraído de un documento PDF.

CONTEXTO DEL DOCUMENTO:
{contexto}

PREGUNTA DEL USUARIO:
{pregunta}

INSTRUCCIONES:
- Responde de forma clara y concisa basándote SOLO en el contexto proporcionado.
- Si la información no está en el contexto, di claramente que no encontraste esa información en el documento.
- Cita brevemente de dónde viene la información cuando sea relevante.
- Responde en el mismo idioma de la pregunta.

RESPUESTA:"""


    modelo_gemini = genai.GenerativeModel('gemini-1.5-flash-latest')
    respuesta = modelo_gemini.generate_content(prompt)
    return respuesta.text

print("Función RAG definida correctamente.")

Función RAG definida correctamente.


In [9]:

def chat_rag(mensaje, historial):
    """Función de chat para la interfaz Gradio."""
    respuesta = responder_pregunta_rag(mensaje)
    historial.append((mensaje, respuesta))
    return "", historial


with gr.Blocks(title="Sistema RAG con Gemini ") as demo:
    gr.Markdown("""
    #  Sistema RAG con Gemini
    ### Pregunta sobre el contenido de tu documento PDF
    Este sistema usa **Retrieval-Augmented Generation (RAG)**:
    1. Busca los fragmentos más relevantes del PDF usando embeddings semánticos
    2. Envía el contexto a Gemini para generar una respuesta precisa
    """)

    chatbot = gr.Chatbot(
        label="Conversación",
        height=400,
        bubble_full_width=False
    )

    with gr.Row():
        txt_input = gr.Textbox(
            placeholder="Escribe tu pregunta sobre el documento...",
            label="Tu pregunta",
            scale=4
        )
        btn_enviar = gr.Button("Enviar 🚀", scale=1, variant="primary")

    btn_limpiar = gr.Button("Limpiar conversación 🗑️")

    gr.Examples(
        examples=[
            ["¿De qué trata este documento?"],
            ["¿Cuáles son los puntos principales?"],
            ["Resume el documento en 3 puntos clave"],
        ],
        inputs=txt_input
    )


    btn_enviar.click(chat_rag, inputs=[txt_input, chatbot], outputs=[txt_input, chatbot])
    txt_input.submit(chat_rag, inputs=[txt_input, chatbot], outputs=[txt_input, chatbot])
    btn_limpiar.click(lambda: [], outputs=[chatbot])

demo.launch(share=True, debug=True)

/tmp/ipykernel_3234/4105187986.py:17: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(
/tmp/ipykernel_3234/4105187986.py:17: DeprecationWarning: The 'bubble_full_width' parameter will be removed in Gradio 6.0. This parameter no longer has any effect.
  chatbot = gr.Chatbot(
/tmp/ipykernel_3234/4105187986.py:17: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://51504363b4f09e5756.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/gradio/queueing.py", line 759, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/gradio/route_utils.py", line 354, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/gradio/blocks.py", line 2191, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/gradio/blocks.py", line 1698, in call_function
    prediction = await anyio.to_thread.run_sync(  # type: ignore
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/anyio/to_thread.py", line 63, in run_sync
    return await get_async_backend().run_sync_in_worker_thread(
           ^^^^^

Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://51504363b4f09e5756.gradio.live


#  Ejercicio 2: Generación de Embeddings y Visualización t-SNE



In [10]:

!pip install -q tensorflow tensorflow-hub bing-image-downloader pillow gradio scikit-learn matplotlib

import os, glob
import numpy as np
import tensorflow as tf
import tensorflow_hub as hub
from PIL import Image
import gradio as gr
print("Librerías cargadas OK")

Librerías cargadas OK


In [11]:

from bing_image_downloader import downloader

categorias = ["dog", "cat", "car", "airplane"]
for cat in categorias:
    downloader.download(cat, limit=5, output_dir='imagenes',
                        adult_filter_off=True, force_replace=False, timeout=60)


rutas_imagenes = []
for cat in categorias:
    rutas_imagenes += sorted(glob.glob(f'imagenes/{cat}/*'))
print(f"Total imágenes descargadas: {len(rutas_imagenes)}")
for r in rutas_imagenes:
    print(" -", r)


[%] Downloading Images to /content/imagenes/dog


[!!]Indexing page: 1

[%] Indexed 17 Images on Page 1.


[%] Downloading Image #1 from https://suchscience.net/wp-content/uploads/2024/03/v2-7u73v-mawkh.jpg
[%] File Downloaded !

[%] Downloading Image #2 from https://animalvivid.com/wp-content/uploads/2023/03/Different-Dog-Breeds-Sitting-on-Grass.jpg
[%] File Downloaded !

[%] Downloading Image #3 from https://cdn.pixabay.com/photo/2023/07/14/11/23/dog-8126822_1280.jpg
[%] File Downloaded !

[%] Downloading Image #4 from https://pawsomeauthority.com/wp-content/uploads/a-collage-of-dog-breed-portraits-pawsome-authority.jpg
[Error]Invalid image, not saving https://pawsomeauthority.com/wp-content/uploads/a-collage-of-dog-breed-portraits-pawsome-authority.jpg

[!] Issue getting: https://pawsomeauthority.com/wp-content/uploads/a-collage-of-dog-breed-portraits-pawsome-authority.jpg
[!] Error:: Invalid image, not saving https://pawsomeauthority.com/wp-content/uploads/a-collage-of-dog-breed-po

In [12]:

URL_MODELO = "https://tfhub.dev/google/tf2-preview/mobilenet_v2/feature_vector/4"
modelo_embed = hub.KerasLayer(URL_MODELO, input_shape=(224, 224, 3), trainable=False)
print(" Modelo MobileNetV2 cargado correctamente desde TF-Hub")


 Modelo MobileNetV2 cargado correctamente desde TF-Hub


In [14]:

def procesar_imagen(ruta_o_pil):
    """Recibe una ruta de imagen o un objeto PIL, lo redimensiona a 224x224 y normaliza a [0,1]."""
    if isinstance(ruta_o_pil, str):
        img = Image.open(ruta_o_pil).convert("RGB")
    else:
        img = ruta_o_pil.convert("RGB")
    img = img.resize((224, 224))
    arr = np.array(img) / 255.0
    arr = np.expand_dims(arr.astype(np.float32), 0)
    return arr

def generar_embedding(ruta_o_pil):
    """Genera el vector embedding (1280-d) de una imagen usando MobileNetV2."""
    arr = procesar_imagen(ruta_o_pil)
    emb = modelo_embed(arr).numpy().flatten()
    return emb


print(f"Generando embeddings para {len(rutas_imagenes)} imágenes...")
embeddings_ref = np.array([generar_embedding(r) for r in rutas_imagenes])
etiquetas = [r.split('/')[-2] for r in rutas_imagenes]
print(" Shape del índice de embeddings:", embeddings_ref.shape)
print("Categorías:", set(etiquetas))


Generando embeddings para 20 imágenes...
 Shape del índice de embeddings: (20, 1280)
Categorías: {'airplane', 'car', 'dog', 'cat'}


In [ ]:

def similitud_coseno(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-10)

def buscar_similares(imagen_pil, top_k=5):
    """Recibe una imagen PIL y devuelve las top_k más parecidas del índice."""
    if imagen_pil is None:
        return []
    emb_q = generar_embedding(imagen_pil)
    sims = np.array([similitud_coseno(emb_q, e) for e in embeddings_ref])
    idx_top = np.argsort(sims)[::-1][:top_k]
    resultados = []
    for i in idx_top:
        categoria = rutas_imagenes[i].split('/')[-2]
        caption = f"{categoria} | sim: {sims[i]:.3f}"
        resultados.append((rutas_imagenes[i], caption))
    return resultados

with gr.Blocks(title="Búsqueda Semántica de Imágenes 🔍") as demo_img:
    gr.Markdown("""
    # 🔍 Búsqueda Semántica de Imágenes
    Sube una imagen y la app encontrará las **5 más parecidas** del índice
    usando embeddings de **MobileNetV2** y **similitud de coseno**.
    """)
    with gr.Row():
        inp = gr.Image(type="pil", label="Imagen de consulta")
        out = gr.Gallery(label="Top-5 más parecidas", columns=5, height="auto")
    btn = gr.Button("Buscar similares 🚀", variant="primary")
    btn.click(buscar_similares, inputs=inp, outputs=out)

demo_img.launch(share=True, debug=True)


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://5bf178b7505f086a35.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:

from sklearn.manifold import TSNE
import matplotlib.pyplot as plt

tsne = TSNE(n_components=2, perplexity=5, random_state=42, init="pca")
proy = tsne.fit_transform(embeddings_ref)

plt.figure(figsize=(9, 7))
colores = {"dog": "tab:red", "cat": "tab:blue", "car": "tab:green", "airplane": "tab:orange"}
for cat in set(etiquetas):
    mask = np.array([e == cat for e in etiquetas])
    plt.scatter(proy[mask, 0], proy[mask, 1],
                label=cat, s=120, alpha=0.8,
                color=colores.get(cat, None))
plt.legend(fontsize=11)
plt.title("Visualización t-SNE de los embeddings (MobileNetV2)", fontsize=13)
plt.xlabel("Componente 1")
plt.ylabel("Componente 2")
plt.grid(alpha=0.3)
plt.show()
